#### 1. Read the data file and create a Dataframe

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev.spark_db.sf_fire_calls (
CallNumber INT, UnitID STRING, IncidentNumber INT, CallType STRING, CallDate DATE,
WatchDate DATE, CallFinalDisposition STRING, AvailableDtTm TIMESTAMP, Address STRING,
City STRING, Zipcode STRING, Battalion STRING, StationArea STRING, Box STRING,
OriginalPriority STRING, Priority STRING, FinalPriority STRING, ALSUnit BOOLEAN,
CallTypeGroup STRING, NumAlarms INT, UnitType STRING, UnitSequenceInCallDispatch INT,
FirePreventionDistrict STRING, SupervisorDistrict STRING, Neighborhood STRING,
Location STRING, RowID STRING, Delay DOUBLE);

In [0]:
raw_fire_df = (
    spark.read.format('csv')
    .option('header', 'true')
    .option('inferSchema','true')
    .load('/Volumes/dev/spark_db/datasets/spark_programming/data/sf-fire-calls.csv')
)

#### 2. Count the records

In [0]:
raw_fire_df.count()

In [0]:
raw_fire_df.display()

#### 3. Verify Dataframe schema with target table(printSchema)

In [0]:
raw_fire_df.printSchema()

#### 4. Transform Dataframe

In [0]:
from pyspark.sql.functions import to_timestamp, expr
fire_df = raw_fire_df.withColumns({
    "AvailableDtTm": to_timestamp("AvailableDtTm", "MM/dd/yyyy hh:mm:ss a"),
    "Zipcode": expr("cast(Zipcode as string)"),
    "FinalPriority": expr("cast(FinalPriority as string)")
                                   })

In [0]:
fire_df.printSchema()

#### 5. Save final dataframe to target table

In [0]:
fire_df.write.mode("overwrite").saveAsTable("dev.spark_db.sf_fire_calls")

In [0]:
%sql
select CallType, Zipcode, count(*) as count from dev.spark_db.sf_fire_calls
where CallType is not Null
group by CallType, Zipcode
order by count desc
limit 3

In [0]:
fires_df = spark.read.table("dev.spark_db.sf_fire_calls")

In [0]:
df1 = fires_df.select("CallType", "Zipcode")
df2 = df1.where("CallType is not Null")
df3 = df2.groupBy("CallType", "Zipcode").count()
df4 = df3.orderBy("count", ascending = False)
df5 = df4.limit(3)

In [0]:
#Apply an Action method
df5.show()

In [0]:
df5.explain(mode="extended")

In [0]:
result_df = (
    fires_df.select("CallType","Zipcode")
    .where("CallType is not Null")
    .groupBy("CallType","Zipcode")
    .count()
    .orderBy("count", ascending = False)
    .limit(3)
)

In [0]:
result_df.display()

In [0]:
#How many distinct types of call were made to Fire Department
#select count(distinct CallType) as distint_call_type_count from dev.spark_db.sf_fire_calls

q1_df = (
    fires_df.selectExpr("count(distinct CallType) as distinct_call_type_count")
)

q1_df.show()

In [0]:
#What were distinct types of calls made to Fire Department
#select distinct CallType as distinct_call_types from dev.spark_db.sf_fire_calls where CallType is not Null

q2_df = (
    fires_df.select("CallType")
    .where("CallType is not Null")
    .distinct()
)
q2_df.show()

In [0]:
#Find out all response for delayed times greater than 5mins
#select CallNumber, Delay from  dev.spark_db.sf_fire_calls where Delay >5

q3_df = (
    fires_df.select("CallNumber", "Delay")
    .where("Delay >5")
)

q3_df.show()

In [0]:
#What were the most common call types
#select CallType, count(*) as count from dev.spark_db.sf_fire_calls where CallType is not Null groupBy CallType orderBy count desc

q4_df = (
    fires_df.selectExpr("CallType")
    .where("CallType is not Null")
    .groupBy("CallType")
    .count()
    .orderBy("count", ascending = False)
)

q4_df.show()

In [0]:
#What Zipcodes were account for most common call types
#select CallType, Zipcode, count(*) as count from dev.spark_db.sf_fire_calls where CallType is not Null group by CallType, Zipcode order by count desc

q5_df = (
    fires_df.select("Zipcode","CallType")
    .where("CallType is not Null")
    .groupBy("Zipcode","CallType").count()
    .orderBy("count", ascending = False)
)

q5_df.show()


In [0]:
#What San Fransico neighborhoods are in the zip code 94102, 94103
#select distinct Neighborhood, Zipcode from dev.spark_db.sf_fire_calls where Zipcode == 94102 or Zipcode == 94103

q6_df = (
    fires_df.selectExpr("Neighborhood")
    .where("Zipcode == 94102 or Zipcode == 9410")
    .distinct()
)

q6_df.show()

In [0]:
#What is the sum of all calls, average, min and max response times of the call
#select sum(NumAlarms), avg(Delay), min(Delay), max(Delay) from dev.spark_db.sf_fire_calls
q7_df = (
    fires_df.selectExpr(
        "sum(NumAlarms)",
        "avg(Delay)",
        "min(Delay)",
        "max(Delay)"
    )
    )

q7_df.show()


In [0]:
#How many distinct years of data is in CSV file
#select distint year(CallDate) as year_num from dev.spark_db.sf_fire_calls order by year_num

q8_df = (
    fires_df.selectExpr("year(CallDate) as year_num")
    .distinct()
    .orderBy("year_num")
)
q8_df.show()

In [0]:
#What week of the year in 2018 had most fire calls
#select weekofyear(CallDate) as week_year, count(*) as count from dev.spark_db.sf_fire_calls where year(CallDate) = 2018 group by week_year order by count desc

q9_df = (
    fires_df.selectExpr("weekofyear(CallDate) as week_year")
    .where("year(CallDate)==2018")
    .groupBy("week_year")
    .count()
    .orderBy("count", ascending = False)
)
q9_df.show()



In [0]:
#What neighborhoods in San Franciso had the worst response time in 2018
#select Neighborhood, Delay from dev.spark_db.sf_fire_calls where year(CallDate) == 2018

q10_df = (
    fires_df.selectExpr("Neighborhood","Delay")
    .where("year(CallDate)==2018")
)
q10_df.show()